In [98]:
# piszemy apke która korzysta z api pushover, jak ktos np na jakiejs stornie gdzie umieszcze bota o cos zapyta, dostane notificationz  serwera pushover
# dodatkowo napiszemy własne tools które zdecyduje kiedy się odpalić  tools napiszemy w json schema 

#import

import json
from dotenv import load_dotenv
from pypdf import PdfReader
import os
from openai import OpenAI
import requests

openai = OpenAI()

In [99]:
# wrzucamy klucze api, token , user i url serwera pushover 

# importujemy klucze api 1 to klucz usera 2 to klucz token
# przekazuejmy je do zmiennych za pomocą nazwy os.getenv i nazwy klucza api
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")

# dodajemy url na który bedziemy wysyłac kod

pushover_url = "https://api.pushover.net/1/messages.json"

In [100]:
# kodujemy funkcje push która zbiera message w argumencie , dane czyli klucze api i url serwera i wysła wiadomosc na serwer a serwer na urządzenie zarejstrowane w pushover api

def push(message):
    dataInfo = {'user' : pushover_user, 'token' : pushover_token, 'message' : message} # te dane są z api w takim formacie jakim musza byc

    # metodą request z metodą post wysyłamy na serwer dane 
    requests.post(pushover_url, data=dataInfo)



In [101]:
push('chuj ci w dupoe')

In [102]:

# piszemy 1 funkcje która bedzie narzedzem tools obsluzy sytuacje w której llm bedzie w stanei odpowiedziec na pytanie usera i zarejstruje to wydarzenie i ruuchmoi pusch i wysle je z danymi 
# funkcje uruchamiaq potem llm w innej funkcji 

# to jest funkncja kóra wykonuej kod

def record_user_details(email, name = 'unknown' , notes =  ' not provided'):
    # uruchaniamy push 
    push(email, name, notes) 

    # zwracamy info do llm ze funkcja uzyta porpawnie
    return{'recorded' : 'ok'}

In [103]:


# piszemy 2 narzędzie tools które obskluzy sytuacje w kórej wiadomosc user nie moze zostac udzielona opdoiwedz bo llm nie ma tego w prompcie
# uruchomiana póxinej w kolejnenj funkcji
 
# to jest funkncja kóra wykonuej kod

def record_unknown_question(question):

    push(question)

    return{'recorded' : 'ok '}

In [104]:

# teraz tworzymy opis tego narzędzia w json, czyli tej funkcji, 

#opis dla modelu, żeby wiedział że ta funkcja istnieje

record_user_details_json = {

    # musi zawierac name czyli nazwe funnkcji tools
    "name" : 'record_user_details',
    # descitpion czyli opis tego co robi
    'desciption' :  "Use this tool to record that a user is interested in being in touch and provided an email address",
    # parametry 
    'parameters' : {
        'type': 'object', # typ obiekt
        'properties' : { # wartosci

            # wypisujemy wszystkie te których uzywa funkcja argumenty  

            "email": {
                "type": "string", # piszemy ich type i description 
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
         },
        "required": ["email"],        # argumenty wymagane
        "additionalProperties": False     
        }
        
    }



In [105]:


record_unknown_question_json = {

    "name" : 'record_unknown_question',
    'description' : "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    'parameters' : {
        "type" : 'object',
        "properties" : {
            'question' : {
                'type' : 'string',
                "description" : "The question that couldn't be answered"
            }
        },
     "required" : ['question'],
      "additionalProperties": False
    }
}

In [106]:
# tworzymy liste tools gdzie łączymy oby 2 instukcje opisy JSON w i liste tools wszystko na podstawie dokuemntacji api OpenAI
# dajemy typ, u nas to funkcja, i definiujemy dfincke dodajac nazwę 


tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [107]:

# piszemy funkcje która w zaelznosci od if else wywoła dane narzędzie czyli funkcje record_unkown_question lub record_user_details 
# funkcja dostaje liste narzędzi jako aprametr czyli cos wymaganego przez api do odpalenie tych 2 funkcji tool_calls to z open ai api / odpala odpowiędnia funkcje z argumentami dla odpowiendiego narżedzia (funkcji)
#wynik pakuje w format OpenAI 
# to llm decyduje jakaktorą funkcje, narzędzie ma wywoloac ta funkcje na pdostawie json schema, bo dzieki niemu wie jak dziala narzędzie

# jako argument rpzeakzujemy wymagania openAI api do uruchomeinia narziędzua funkcji + json schema
def handle_tools_call(tool_calls):
    # tworzymy zmienną gdzie przekazemy wyniki narzedzi czyli to co zwracają funnkcje unknown i recorded details, czyli info dla api Open AI {result ok}
    results = []
    # zapetlamy tablice z narzdedziami tool_calls, tool call to pojedyncze nardzedzie w danej petli
    for tool_call in tool_calls:
        # wyciagamy nazwe narzędzia któr jest zapętlone
        tool_name = tool_call.function.name
        # pobieramy argumenty znajdujące się w properties json shcema dla narziędza
        arguments = json.loads(tool_call.function.arguments)

        # 1 if sprawdza czy tool_name to danna funkcja, narziędzie, jesli tak to je odpala
        if tool_name == 'record_user_details':

            # jesli if true i odpalone przez llm narziędzie to user_details to odpalamy funklcje ** rozpakowujemy argumenty i przekazujemy wynik funckjju do result
            # argumenty pobieramy wyzej
            result = record_user_details(**arguments)
        
        elif tool_name == 'record_unknown_question':

            result = record_unknown_question(**arguments)

        #dodajemy wynik do results w formacie jakiego wymaga openAI api role tool oznacza ze to wyniok narzędzia content to wunik funkcji tool_call_id to id wykonania

    return results.append({'role' : 'tool', 'content' : json.dumps(result) , 'tool_call_id' : tool_call.id  })


In [108]:
# pobieramy pdf oraz txt

reader = PdfReader("linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Ed Donner"


In [109]:
# dajemy prompt 

system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [110]:
# funkcja która odpala wszystkie inne i przyjmuje history oraz message czyli tablice historii  oraz message 
from ast import While

from langchain_core.messages.tool import tool_call


def chat(message,history):

    #budujemy messages liste message, prompt system oraz history

    messages = [{'role':'system','content':system_prompt}] + history + [{'role':'user','content':message}]

    # ustaweiamy flage done na false, flaga zamenia się na true i false potem w kodzie

    done = False

    # petla while ktora wykinuje sie dopoki done jest falsw zatrzymuje sue jak done zamieni się na true

    while not done:

            # tworzymy zmienną response ktora otrzuje odpowiedz od modelu llm open ai ktory wysyla liste message, model oraz liste narzędzi
            response = openai.chat.completions.create(  model = 'gpt-4.1-mini', messages=messages, tools=tools )

            # tworzymy zmienną finish reason do której zwracamy powód zakonczenia odpowiedzi np tool_calls czyli wywloane narzedzie lub stop, normalna odpowiedz
            finish_reason = response.choices[0].finish_reason
            
            # tworzymy warunek if jesli powod zakonczenia odpowiedzi to tool_calls czyli wywolanie narzędzia

            if finish_reason == 'tool_calls':


                # do zmiennej message wyciągamy waidomosc z api llma

                message = response.choices[0].message

                # wyciagamy lite narzędzi jakich api uzyło

                tool_calls = message.tool_calls 

                # do zmiennej rezult rpzakzujemy wynik funkcji handle_tools_calls która wywojumey z argumentem narzędzi listy

                result = handle_tools_call(tool_calls)

                # dodajemy do messages info od medolu jakie nardzędzie chciał wywłoać

                messages.append(message)

                # dodaje wynik narzędzia funkcji handle tool cas przeakzanej do results do listy messages
                messages.extend(result)

            else:
                # jesli finiszh reason inny niz wywolane narzedziem to zmaienimy done na true i konczymy petle while
                done = True

                # i zwracamy odpowiedz bez narzedzi

    return response.choices[0].message.content




In [111]:
history = []

msg1 = 'czym się zajmujesz'

answer1 = chat(msg1,history)

print(answer1)

Cześć! Jestem Ed Donner, przedsiębiorcą, inżynierem oprogramowania i data scientistem. Obecnie jestem współzałożycielem i CTO w Nebula.io, gdzie rozwijamy rozwiązania oparte na generatywnej sztucznej inteligencji (Gen AI) i własnych modelach językowych (LLM), które pomagają rekruterom efektywniej wyszukiwać, rozumieć i angażować talenty. Moja praca ma na celu pomaganie ludziom w odkrywaniu ich potencjału i znalezieniu satysfakcjonującej pracy, zgodnej z ich pasjami i umiejętnościami.

Wcześniej spędziłem wiele lat w JPMorgan, gdzie zarządzałem zespołami technologicznymi i rozwijałem oprogramowanie dla rynków finansowych na całym świecie.

Po godzinach uwielbiam eksperymentować z dużymi modelami językowymi i dzielić się wiedzą podczas wykładów na temat AI i inżynierii LLM.

Jeśli chcesz dowiedzieć się więcej lub porozmawiać, chętnie odpowiem!
